# Stellar Chromospheric Activity — Implementation / 구현
# 항성 채층 활동 — 구현

**Paper**: Hall, J.C., "Stellar Chromospheric Activity", *Living Reviews in Solar Physics*, 5, 2 (2008)
**DOI**: 10.12942/lrsp-2008-2

This notebook implements the key chromospheric activity diagnostics described in the paper:
이 노트북은 논문에 기술된 핵심 채층 활동 진단법을 구현합니다:

1. **Mt. Wilson S index** — Ca II H & K 방출 측정 / Ca II H & K emission measurement
2. **$R'_{\rm HK}$ calibration** — 광구 기여 제거 / Photospheric contribution removal
3. **Wilson-Bappu effect** — Ca II K 방출폭-광도 관계 / Ca II K emission width-luminosity relation
4. **Flux-flux relationships** — 채층 진단선 간 상관관계 / Inter-diagnostic correlations
5. **Vaughan-Preston Gap** — 활동 분포의 이봉 구조 / Bimodal activity distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib import cm

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## Part 1: Mt. Wilson S Index / Mt. Wilson S 지수

The Mt. Wilson HK photometer measures Ca II H (3968.47 Å) and K (3933.66 Å) emission using triangular bandpasses of 1.09 Å FWHM centered on each line, plus two 20 Å pseudo-continuum bandpasses (R at 4001 Å and V at 3901 Å).

Mt. Wilson HK 측광기는 각 선의 중심에 1.09 Å FWHM 삼각 대역통과로 Ca II H (3968.47 Å)와 K (3933.66 Å) 방출을 측정하며, 두 개의 20 Å 의사 연속 대역(R: 4001 Å, V: 3901 Å)을 사용합니다.

$$S = \alpha \frac{N_H + N_K}{N_R + N_V}$$

where $\alpha$ is an instrumental calibration factor (~2.4 for the original HKP-2).
여기서 $\alpha$는 기기 보정 계수입니다 (원래 HKP-2에서 ~2.4).

In [ ]:
def compute_s_index(n_h: float, n_k: float, n_r: float, n_v: float,
                    alpha: float = 2.4) -> float:
    """Compute the Mt. Wilson S index from HK photometer counts.

    Args:
        n_h: Counts in the Ca II H bandpass (1.09 Å FWHM at 3968.47 Å).
        n_k: Counts in the Ca II K bandpass (1.09 Å FWHM at 3933.66 Å).
        n_r: Counts in the R pseudo-continuum bandpass (20 Å at 4001 Å).
        n_v: Counts in the V pseudo-continuum bandpass (20 Å at 3901 Å).
        alpha: Instrumental calibration factor (default: 2.4 for HKP-2).

    Returns:
        S index (dimensionless ratio).
    """
    return alpha * (n_h + n_k) / (n_r + n_v)


# Example: synthetic HK photometer measurement for a Sun-like star
# The Sun has S_MW ≈ 0.171 (cycle mean)
np.random.seed(42)
n_r_sun, n_v_sun = 1000.0, 950.0  # pseudo-continuum counts
# Adjust H & K counts to reproduce S ≈ 0.171
n_hk_sum = 0.171 * (n_r_sun + n_v_sun) / 2.4  # ≈ 138.9
n_h_sun = n_hk_sum * 0.48  # H is slightly weaker than K
n_k_sun = n_hk_sum * 0.52

s_sun = compute_s_index(n_h_sun, n_k_sun, n_r_sun, n_v_sun)
print(f"Solar S index (cycle mean): {s_sun:.3f}")
print(f"  N_H = {n_h_sun:.1f}, N_K = {n_k_sun:.1f}")
print(f"  N_R = {n_r_sun:.1f}, N_V = {n_v_sun:.1f}")
print(f"  Typical range: S_min ≈ 0.164 (activity minimum) to S_max ≈ 0.178 (activity maximum)")

## Part 2: S-to-$R'_{\rm HK}$ Conversion / S 지수에서 $R'_{\rm HK}$로 변환

The $S$ index includes photospheric flux contamination that varies with spectral type ($B-V$ color). To isolate the chromospheric contribution, Noyes et al. (1984) defined:

$S$ 지수에는 분광형($B-V$ 색)에 따라 변하는 광구 플럭스 오염이 포함됩니다. 채층 기여만을 분리하기 위해 Noyes et al. (1984)이 정의한 것:

$$R'_{\rm HK} = R_{\rm HK} - R_{\rm phot}$$

where $R_{\rm HK} = C_{\rm cf}(B-V) \cdot S$ converts $S$ to surface flux ratio, and $R_{\rm phot}$ is the photospheric correction.

### Calibration coefficients (Rutten 1984, Noyes et al. 1984):
### 보정 계수 (Rutten 1984, Noyes et al. 1984):

$$\log C_{\rm cf} = 0.25(B-V)^3 - 1.33(B-V)^2 + 0.43(B-V) + 0.24$$

$$\log R_{\rm phot} = -4.898 + 1.918(B-V)^2 - 2.893(B-V)^3$$

Valid for $0.44 \leq (B-V) \leq 0.82$ (F2V to K2V main sequence).
유효 범위: $0.44 \leq (B-V) \leq 0.82$ (F2V에서 K2V 주계열).

In [ ]:
def log_ccf(bv: np.ndarray) -> np.ndarray:
    """Conversion factor from S to surface flux ratio (Rutten 1984).

    Args:
        bv: B-V color index.

    Returns:
        log10(C_cf).
    """
    return 0.25 * bv**3 - 1.33 * bv**2 + 0.43 * bv + 0.24


def log_r_phot(bv: np.ndarray) -> np.ndarray:
    """Photospheric contribution to R_HK (Noyes et al. 1984).

    Args:
        bv: B-V color index.

    Returns:
        log10(R_phot).
    """
    return -4.898 + 1.918 * bv**2 - 2.893 * bv**3


def s_to_r_prime_hk(s: float, bv: float) -> float:
    """Convert Mt. Wilson S index to R'_HK.

    Args:
        s: Mt. Wilson S index.
        bv: B-V color index.

    Returns:
        R'_HK (chromospheric emission ratio, typically 1e-5 to 1e-4).
    """
    c_cf = 10**log_ccf(bv)
    r_hk = c_cf * s
    r_phot = 10**log_r_phot(bv)
    return r_hk - r_phot


# Demonstrate for the Sun (B-V = 0.656)
bv_sun = 0.656
r_prime_sun = s_to_r_prime_hk(s_sun, bv_sun)
print(f"Sun: S = {s_sun:.3f}, B-V = {bv_sun}")
print(f"  C_cf = {10**log_ccf(bv_sun):.4e}")
print(f"  R_HK = {10**log_ccf(bv_sun) * s_sun:.4e}")
print(f"  R_phot = {10**log_r_phot(bv_sun):.4e}")
print(f"  R'_HK = {r_prime_sun:.4e}")
print(f"  log R'_HK = {np.log10(r_prime_sun):.3f}")
print(f"  (Expected: log R'_HK ≈ -4.94 for the Sun)")

## Part 3: Wilson-Bappu Effect / Wilson-Bappu 효과

Wilson & Bappu (1957) discovered a tight correlation between the width of the Ca II K emission core ($W_0$) and absolute visual magnitude ($M_V$), independent of spectral type:

Wilson & Bappu (1957)는 Ca II K 방출 코어의 폭($W_0$)과 절대 시등급($M_V$) 사이에 분광형과 무관한 강한 상관관계를 발견했습니다:

$$M_V = 27.59 - 14.94 \log W_0({\rm K})$$

This makes the Ca II K line width a powerful **distance indicator** for late-type stars.
이로써 Ca II K 선폭은 만기형 별의 강력한 **거리 지시자**가 됩니다.

In [ ]:
def wilson_bappu(w0_km_s: float) -> float:
    """Compute absolute magnitude from Ca II K emission width.

    Args:
        w0_km_s: Ca II K emission core width in km/s.

    Returns:
        Absolute visual magnitude M_V.
    """
    return 27.59 - 14.94 * np.log10(w0_km_s)


# Wilson-Bappu relation across stellar luminosity classes
w0_range = np.linspace(15, 200, 200)  # km/s
mv_range = wilson_bappu(w0_range)

# Reference stars (approximate values from the literature)
stars = {
    "Sun (G2V)": (30, 4.83),
    "Procyon (F5IV-V)": (42, 2.68),
    "Arcturus (K2III)": (65, -0.31),
    "Aldebaran (K5III)": (72, -0.63),
    "Canopus (F0Ib)": (150, -5.53),
}

fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(np.log10(w0_range), mv_range, 'b-', linewidth=2,
        label=r"$M_V = 27.59 - 14.94 \log W_0$")

for name, (w0, mv) in stars.items():
    ax.plot(np.log10(w0), mv, 'o', markersize=10, zorder=5)
    ax.annotate(name, (np.log10(w0), mv), textcoords="offset points",
                xytext=(10, 5), fontsize=9)

ax.set_xlabel(r"$\log W_0(K)$ [km/s]", fontsize=13)
ax.set_ylabel(r"$M_V$ [mag]", fontsize=13)
ax.set_title("Wilson-Bappu Effect: Ca II K Emission Width vs. Luminosity\n"
             "Wilson-Bappu 효과: Ca II K 방출폭 대 광도", fontsize=13)
ax.invert_yaxis()  # brighter = lower M_V
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("\nKey insight: The relation spans ~15 magnitudes in luminosity,")
print("from dwarfs (M_V ~ +5) to supergiants (M_V ~ -8),")
print("making Ca II K width a powerful distance indicator.")
print("\n핵심 통찰: 이 관계는 광도에서 ~15등급 범위를 포괄하며,")
print("왜성(M_V ~ +5)에서 초거성(M_V ~ -8)까지 적용됩니다.")

## Part 4: Vaughan-Preston Gap & Activity Distribution / Vaughan-Preston 간극과 활동 분포

Vaughan & Preston (1980) discovered that the distribution of chromospheric activity among solar-type stars is **bimodal**: there is a gap (the "Vaughan-Preston Gap") separating "active" young stars from "inactive" old stars at approximately $\log R'_{\rm HK} \approx -4.75$.

Vaughan & Preston (1980)은 태양형 별의 채층 활동 분포가 **이봉(bimodal)**임을 발견했습니다: "활동적" 젊은 별과 "비활동적" 오래된 별을 분리하는 간극(Vaughan-Preston Gap)이 $\log R'_{\rm HK} \approx -4.75$에 존재합니다.

This bimodality is now understood as reflecting the rapid decline of magnetic activity as stars spin down via magnetized winds (Skumanich 1972 law: activity ∝ $t^{-1/2}$).

이 이봉 구조는 자기화된 항성풍에 의한 회전 감속에 따른 자기 활동의 급격한 감소를 반영하는 것으로 이해됩니다 (Skumanich 1972 법칙: 활동 ∝ $t^{-1/2}$).

In [ ]:
# Simulate a stellar sample showing the Vaughan-Preston Gap
np.random.seed(123)

# Two populations: young active + old inactive
n_active = 80
n_inactive = 150
n_vp_gap = 15  # sparse population in the gap

# Active stars: log R'_HK ~ -4.2 to -4.6
log_rhk_active = np.random.normal(-4.4, 0.12, n_active)
bv_active = np.random.uniform(0.45, 0.90, n_active)

# Inactive stars: log R'_HK ~ -4.85 to -5.15
log_rhk_inactive = np.random.normal(-5.0, 0.08, n_inactive)
bv_inactive = np.random.uniform(0.45, 0.90, n_inactive)

# Gap region: sparse
log_rhk_gap = np.random.uniform(-4.65, -4.85, n_vp_gap)
bv_gap = np.random.uniform(0.50, 0.85, n_vp_gap)

# Combined
all_log_rhk = np.concatenate([log_rhk_active, log_rhk_inactive, log_rhk_gap])
all_bv = np.concatenate([bv_active, bv_inactive, bv_gap])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: S vs B-V (like Vaughan & Preston's original figure)
# Convert back to S for display
all_s = np.array([10**log_ccf(bv) for bv in all_bv])
all_s = (10**all_log_rhk + 10**np.array([log_r_phot(bv) for bv in all_bv])) / all_s

colors = np.where(all_log_rhk > -4.75, 'red', np.where(all_log_rhk < -4.85, 'blue', 'gray'))
ax1.scatter(all_bv, all_s, c=colors, alpha=0.5, s=20)
ax1.axhline(y=0.2, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel("B - V", fontsize=13)
ax1.set_ylabel("S index", fontsize=13)
ax1.set_title("S index vs. Color\nS 지수 대 색지수", fontsize=13)
ax1.legend(handles=[
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=8, label='Active / 활동적'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=8, label='Inactive / 비활동적'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=8, label='VP Gap / VP 간극'),
])

# Right: histogram of log R'_HK showing bimodality
ax2.hist(all_log_rhk, bins=40, range=(-5.3, -4.0), color='steelblue',
         edgecolor='white', alpha=0.8)
ax2.axvline(x=-4.75, color='red', linestyle='--', linewidth=2, label="VP Gap boundary")
ax2.axvline(x=np.log10(r_prime_sun), color='orange', linestyle='-', linewidth=2,
            label=f"Sun (log R'_HK = {np.log10(r_prime_sun):.2f})")
ax2.set_xlabel(r"$\log R'_{\rm HK}$", fontsize=13)
ax2.set_ylabel("Number of stars / 별 수", fontsize=13)
ax2.set_title("Vaughan-Preston Gap: Bimodal Activity Distribution\n"
              "Vaughan-Preston 간극: 이봉 활동 분포", fontsize=13)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("The gap at log R'_HK ≈ -4.75 separates:")
print("  Active (young, <2 Gyr): log R'_HK > -4.75")
print("  Inactive (old, >2 Gyr): log R'_HK < -4.85")
print(f"  Sun sits among the inactive population: log R'_HK ≈ {np.log10(r_prime_sun):.2f}")

## Part 5: Flux-Flux Relationships / 플럭스-플럭스 관계

Schrijver (1987) established that chromospheric and coronal emission fluxes follow power-law relationships after subtracting a "basal" flux component $\phi$:

Schrijver (1987)는 "기저(basal)" 플럭스 성분 $\phi$를 빼면 채층과 코로나 방출 플럭스가 멱법칙 관계를 따른다는 것을 확립했습니다:

$$\log(\mathcal{F}_i - \phi_i) = a \log(\mathcal{F}_j - \phi_j) + b$$

The basal flux represents the minimum emission level present even in the absence of magnetic activity (acoustic heating). Key relationships include Ca II ↔ Mg II ↔ X-ray.

기저 플럭스는 자기 활동이 없어도 존재하는 최소 방출 수준을 나타냅니다(음향 가열). 핵심 관계에는 Ca II ↔ Mg II ↔ X선이 포함됩니다.

In [ ]:
def flux_flux_relation(f_j: np.ndarray, phi_j: float, a: float, b: float,
                       phi_i: float) -> np.ndarray:
    """Compute flux-flux relation (Schrijver 1987).

    Args:
        f_j: Surface flux of diagnostic j (erg cm^-2 s^-1).
        phi_j: Basal flux of diagnostic j.
        a: Power-law slope.
        b: Power-law intercept.
        phi_i: Basal flux of diagnostic i.

    Returns:
        Surface flux of diagnostic i.
    """
    excess_j = np.maximum(f_j - phi_j, 1e-10)
    log_excess_i = a * np.log10(excess_j) + b
    return 10**log_excess_i + phi_i


# Approximate parameters from Schrijver (1987) and Rutten et al. (1991)
# Ca II K surface flux range for solar-type stars
f_ca = np.logspace(5.0, 7.0, 200)  # erg cm^-2 s^-1

# Basal fluxes (approximate, B-V ~ 0.65)
phi_ca = 2e5   # Ca II K basal flux
phi_mg = 3e5   # Mg II k basal flux
phi_x = 0.0    # X-ray basal flux (effectively zero)

# Ca II → Mg II: nearly linear (slope ~ 1)
a_ca_mg, b_ca_mg = 1.0, 0.6
f_mg = flux_flux_relation(f_ca, phi_ca, a_ca_mg, b_ca_mg, phi_mg)

# Ca II → X-ray: steeper power law (slope ~ 1.5-2)
a_ca_x, b_ca_x = 1.67, -3.5
f_x = flux_flux_relation(f_ca, phi_ca, a_ca_x, b_ca_x, phi_x)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Ca II vs Mg II
ax1.loglog(f_ca - phi_ca, f_mg - phi_mg, 'b-', linewidth=2)
ax1.set_xlabel(r"$\mathcal{F}_{Ca\,II} - \phi_{Ca}$ [erg cm$^{-2}$ s$^{-1}$]", fontsize=12)
ax1.set_ylabel(r"$\mathcal{F}_{Mg\,II} - \phi_{Mg}$ [erg cm$^{-2}$ s$^{-1}$]", fontsize=12)
ax1.set_title("Ca II ↔ Mg II Flux Relation\n(slope ≈ 1.0, nearly linear)", fontsize=12)
ax1.annotate("Sun", xy=(8e5, 1.5e6), fontsize=11, color='orange',
             arrowprops=dict(arrowstyle='->', color='orange'),
             xytext=(2e6, 5e6))

# Ca II vs X-ray
ax2.loglog(f_ca - phi_ca, f_x, 'r-', linewidth=2)
ax2.set_xlabel(r"$\mathcal{F}_{Ca\,II} - \phi_{Ca}$ [erg cm$^{-2}$ s$^{-1}$]", fontsize=12)
ax2.set_ylabel(r"$\mathcal{F}_X$ [erg cm$^{-2}$ s$^{-1}$]", fontsize=12)
ax2.set_title("Ca II → X-ray Flux Relation\n(slope ≈ 1.67, nonlinear)", fontsize=12)

plt.tight_layout()
plt.show()

print("Key insight from flux-flux relations:")
print("  • Ca II ↔ Mg II: nearly linear (slope ~1) → both trace similar magnetic heating")
print("  • Ca II → X-ray: steep power law (slope ~1.7) → coronal heating amplifies")
print("    magnetic flux variations more than chromospheric heating")
print("\n플럭스-플럭스 관계의 핵심 통찰:")
print("  • Ca II ↔ Mg II: 거의 선형 → 유사한 자기 가열 과정 추적")
print("  • Ca II → X선: 급경사 멱법칙 → 코로나 가열이 자기 플럭스 변화를 증폭")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| S index | Mt. Wilson HKP-2 photometer (1966-2003) | Automated spectroscopic surveys (e.g., HARPS, HIRES) |
| $R'_{\rm HK}$ | Noyes et al. (1984) calibration | Updated calibrations by Mittag et al. (2013), Lorenzo-Oliveira et al. (2018) |
| Wilson-Bappu | $M_V = 27.59 - 14.94 \log W_0$ | Refined with Hipparcos/Gaia parallaxes |
| Flux-flux relations | Schrijver (1987) power laws | Extended to UV, FUV with HST; ALMA mm continuum |
| VP Gap | $\log R'_{\rm HK} \approx -4.75$ bimodality | Confirmed with Kepler rotation periods; related to "stalled spin-down" |
| Activity-age | Skumanich $t^{-1/2}$ decay | Gyrochronology (Barnes 2007); weakened braking beyond solar age (van Saders et al. 2016) |